In [2]:
import pandas as pd
import numpy as np

In [4]:
metrics = pd.read_csv(
    "/home/nathan/Documents/Scolaire/7_Cesure/2_KU_Leuven/methyldl/output/cancer_detector/uniform/calibration/calibration_summary.csv"
)

In [12]:
deconv_map = {"xgb": "XGB", "mlp": "MLP", "swn": "SWN", "nnls": "NNLS", "psls": "PSLS"}
calib_map = {
    "uncalibrated": "None",
    "linear_clip0_normalize": "Lin.\\ clip0+norm",
    "linear_simplex_projection": "Lin.\\ simplex",
    "vector_scaling": "Vec.\\ scaling",
}

deconv_order = ["xgb", "mlp", "swn", "nnls", "psls"]
calib_order = list(calib_map.keys())
n_calib = len(calib_order)
n_total = len(deconv_order) * n_calib

lines = []
first_row = True
for i, dec in enumerate(deconv_order):
    for j, cal in enumerate(calib_order):
        row = metrics[
            (metrics["deconvolver"] == dec) & (metrics["calibration_method"] == cal)
        ].iloc[0]

        r2 = f"{row['overall_r2'] * 100:.2f}"
        loa = f"[{row['loa_lower']*1e2:.2f}, {row['loa_upper']*1e2:.2f}]"
        loa_worst = f"[{row['worst_class_loa_lower']*1e2:.2f}, {row['worst_class_loa_upper']*1e2:.2f}]"
        mae = f"{row['mae']*1e3:.2f}"
        mse = f"{row['mse']*1e4:.2f}"
        kl = f"{row['kl']*1e2:.2f}"

        cal_label = calib_map[cal]

        if first_row:
            lines.append(
                f"                             & \\multirow{{{n_total}}}{{*}}{{Uniform}}"
            )
            lines.append(
                f"                             & \\multirow{{{n_calib}}}{{*}}{{{deconv_map[dec]}}}"
                f"              & {cal_label:<25s} & {r2:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae:<12s} & {mse:<12s} & {kl} \\\\"
            )
            first_row = False
        elif j == 0:
            lines.append(
                f"                             &                                   "
                f"& \\multirow{{{n_calib}}}{{*}}{{{deconv_map[dec]}}} & {cal_label:<25s} & {r2:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae:<12s} & {mse:<12s} & {kl} \\\\"
            )
        else:
            lines.append(
                f"                             &                                   "
                f"&                       & {cal_label:<25s} & {r2:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae:<12s} & {mse:<12s} & {kl} \\\\"
            )

    if i < len(deconv_order) - 1:
        lines.append("        \\cmidrule(l){3-10}")

print("\n".join(lines))

                             & \multirow{20}{*}{Uniform}
                             & \multirow{4}{*}{XGB}              & None                      & 97.00             & [-3.09, 3.09]       & [-8.13, 8.70]           & 3.97         & 2.48         & 7.51 \\
                             &                                   &                       & Lin.\ clip0+norm          & 97.37             & [-2.89, 2.89]       & [-7.95, 8.12]           & 3.67         & 2.17         & 7.61 \\
                             &                                   &                       & Lin.\ simplex             & 97.82             & [-2.63, 2.63]       & [-7.92, 7.96]           & 3.31         & 1.81         & 8.55 \\
                             &                                   &                       & Vec.\ scaling             & 97.35             & [-2.90, 2.90]       & [-6.86, 6.94]           & 4.16         & 2.20         & 5.83 \\
        \cmidrule(l){3-10}
                             &          